# NB09 — Multicollinearity

> **StatQuest: "When two predictors tell the same story, the model can't figure out which one to credit."**

---

## The main ideas are:

1. Multicollinearity = two or more predictors are highly correlated with each other
2. This does NOT ruin predictions — R^2 is unaffected
3. This DOES ruin coefficient interpretation — SEs blow up, t-tests become unreliable
4. VIF (Variance Inflation Factor) measures how bad it is
5. Fixes: drop one predictor, use Ridge, use PCA first


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

def flow_diagram(steps, title, colors=None, notes=None, figsize=(14, 2.8)):
    n = len(steps)
    default_colors = ['#1565C0','#2E7D32','#E65100','#6A1B9A',
                      '#00695C','#AD1457','#37474F','#4E342E',
                      '#0277BD','#558B2F','#C62828','#F57F17']
    colors = (colors or default_colors)[:n]
    notes  = notes or ['']*n
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.3, n*3.1); ax.set_ylim(-1.2, 2.4); ax.axis('off')
    bw, bh = 2.6, 1.3
    for i,(step,color,note) in enumerate(zip(steps,colors,notes)):
        x = i*3.1
        box = FancyBboxPatch((x,0.2),bw,bh,boxstyle="round,pad=0.12",
                             facecolor=color,edgecolor='white',linewidth=1.5,alpha=0.90)
        ax.add_patch(box)
        ax.text(x+bw/2,0.2+bh/2,step,ha='center',va='center',fontsize=8.5,
                color='white',fontweight='bold',multialignment='center')
        if note:
            ax.text(x+bw/2,0.02,note,ha='center',va='top',fontsize=7,
                    color='#555',style='italic')
        if i < n-1:
            ax.annotate('',xy=(x+bw+0.38,0.2+bh/2),xytext=(x+bw+0.08,0.2+bh/2),
                       arrowprops=dict(arrowstyle='->',color='#444',lw=2.2))
    ax.set_title(title,fontsize=11,fontweight='bold',pad=6,color='#222')
    plt.tight_layout(pad=0.4); plt.show()

flow_diagram(
    steps=[
        'Suspected\nmulticollinearity\n(correlated X)',
        'Check correlation\nheatmap',
        'Compute VIF\nfor each X_j\n(1 / (1 - R^2_j))',
        'VIF > 10?\nSevere problem',
        'Diagnose:\nSEs large?\nt-stats low\nbut R^2 high?',
        'Fix options:\nDrop one X,\nRidge, PCA,\nmore data',
        'Re-check VIF\nafter fix',
    ],
    title='NB09 Conceptual Flow: Detecting and Fixing Multicollinearity',
    colors=['#C62828','#1565C0','#2E7D32','#E65100','#6A1B9A','#00695C','#37474F'],
    figsize=(16, 2.8),
)


## Why multicollinearity hurts interpretation

Imagine you want to know the effect of x1 on y, but x1 and x2 are nearly identical.
The model says: "I can explain y equally well by crediting x1, or x2, or half-and-half."
This ambiguity makes both coefficients unstable and their SEs huge.

**Key insight (StatQuest):**
> "Predictions are fine. Interpretations break."


In [ ]:
import numpy as np
import statsmodels.api as sm

np.random.seed(42)
n = 100
x1 = np.random.normal(0, 1, n)
x2 = 0.97*x1 + np.random.normal(0, 0.2, n)   # nearly identical to x1
y  = 3*x1 + 2*x2 + np.random.normal(0, 1, n)

print(f"Correlation(x1, x2) = {np.corrcoef(x1,x2)[0,1]:.3f}")
print()

# Individually: both are significant and correctly estimated
for name, xi in [('x1 only', x1), ('x2 only', x2)]:
    Xsm = sm.add_constant(xi)
    res = sm.OLS(y, Xsm).fit()
    print(f"--- {name} ---")
    print(f"  coef = {res.params[1]:.3f}  SE = {res.bse[1]:.3f}  p = {res.pvalues[1]:.4f}")

print()
# Together: SEs explode, both become "insignificant"
Xsm_both = sm.add_constant(np.column_stack([x1, x2]))
res_both = sm.OLS(y, Xsm_both).fit()
print("--- x1 AND x2 together ---")
print(f"  b1: coef = {res_both.params[1]:.3f}  SE = {res_both.bse[1]:.3f}  p = {res_both.pvalues[1]:.4f}")
print(f"  b2: coef = {res_both.params[2]:.3f}  SE = {res_both.bse[2]:.3f}  p = {res_both.pvalues[2]:.4f}")
print(f"  R^2 = {res_both.rsquared:.4f}  (unchanged — predictions are fine)")
print()
print("Notice: R^2 is high but individual p-values are not significant.")
print("This is the multicollinearity symptom.")


In [ ]:
# VIF — Variance Inflation Factor
# VIF_j = 1 / (1 - R^2_j)
# where R^2_j = R^2 from regressing x_j on all other x's

import numpy as np
import statsmodels.api as sm

def compute_vif(X_arr, names):
    n, k = X_arr.shape
    print(f"{'Feature':<15} {'VIF':>8}  Interpretation")
    print("-"*50)
    for j, name in enumerate(names):
        y_j  = X_arr[:,j]
        X_oth = np.delete(X_arr, j, axis=1)
        Xd    = sm.add_constant(X_oth)
        r2_j  = sm.OLS(y_j, Xd).fit().rsquared
        vif   = 1 / max(1-r2_j, 1e-9)
        interp = ('OK' if vif < 5 else
                  'Moderate' if vif < 10 else 'SEVERE -- action needed')
        print(f"{name:<15} {vif:>8.2f}  {interp}")

X_mat = np.column_stack([x1, x2])
print("VIF analysis:")
compute_vif(X_mat, ['x1','x2'])

print()
# VIF on real housing data
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
print("\nCalifornia Housing VIF:")
compute_vif(housing.data, list(housing.feature_names))


In [ ]:
# Visualise the collinearity problem geometrically
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 80

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, corr, title in zip(axes,
    [0.0, 0.7, 0.97],
    ['No collinearity (r=0.0)', 'Moderate (r=0.7)', 'Severe (r=0.97)']):
    x1 = np.random.normal(0,1,n)
    x2 = corr*x1 + np.sqrt(1-corr**2)*np.random.normal(0,1,n)
    y  = 3*x1 + 2*x2 + np.random.normal(0,1,n)
    X  = sm.add_constant(np.column_stack([x1,x2]))
    res = sm.OLS(y,X).fit()
    se1, se2 = res.bse[1], res.bse[2]
    ax.scatter(x1, x2, s=15, alpha=0.5, color='steelblue')
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.set_title(f'{title}\nSE(b1)={se1:.2f}  SE(b2)={se2:.2f}')
    ax.grid(alpha=0.3)

plt.suptitle('Higher correlation between predictors -> larger standard errors -> less reliable t-tests',
             fontsize=11)
plt.tight_layout(); plt.show()


## Key Takeaways

| Symptom | What it means |
|---------|--------------|
| VIF > 10 | Severe multicollinearity — take action |
| High R^2, low t-stats | Classic multicollinearity fingerprint |
| Coefficients change sign when predictor added/removed | Severe multicollinearity |
| Predictions stable, interpretations unstable | Multicollinearity confirmed |

**Fixes ranked by simplicity:**
1. Drop one of the correlated predictors (if domain knowledge allows)
2. Collect more data (reduces SE naturally)
3. Ridge regression (NB11) — penalises large coefficients, stabilises SEs
4. PCA on correlated predictors, then regress on components

**Next: NB10 — polynomial regression and interaction terms.**
